In [33]:
from astropy.io import fits
import pandas as pd
import numpy as np
file1 = r"C:\Users\invet\OneDrive\Desktop\Inter-IIT\Datasets\ch2_xsm_20200529_v1\xsm\data\2020\05\29\raw\ch2_xsm_20200529_v1_level1.fits"
file2 = r"C:\Users\invet\OneDrive\Desktop\Inter-IIT\Datasets\ch2_xsm_20210828_v1\xsm\data\2021\08\28\raw\ch2_xsm_20210828_v1_level1.fits"
hdul = fits.open(file1)
START_UTC = "2020-05-29T10:40:00"
END_UTC = "2020-05-29T10:43:00"
TARGET_LAT = 57.51
TARGET_LON = 78.14

In [32]:
# Create TWO time-filtered datasets from file1 and file2
# Dataset 1 window comes from START_UTC/END_UTC (Cell 1)
# Dataset 2 window is fixed as requested

START_UTC_2 = "2021-08-28T12:53:05"
END_UTC_2 = "2021-08-28T12:54:41"


def fits_to_dataframe(fits_path: str) -> pd.DataFrame:
    with fits.open(fits_path) as hdul_local:
        data_local = hdul_local[1].data

        clean = {}
        for nm in data_local.columns.names:
            col = data_local[nm]
            if col.ndim == 1:
                clean[nm] = col

    out = pd.DataFrame(clean)

    # Convert FITS big-endian numeric arrays to native-endian for pandas ops
    for c in out.columns:
        arr = out[c].to_numpy(copy=False)
        if arr.dtype.byteorder == ">":
            out[c] = arr.byteswap().view(arr.dtype.newbyteorder("="))

    return out


def add_utc_column(df_in: pd.DataFrame) -> pd.DataFrame:
    d = df_in.copy()
    if "UTC" in d.columns:
        d["_UTC"] = pd.to_datetime(d["UTC"], utc=True, errors="coerce")
    elif "UTCSTRING" in d.columns:
        d["_UTC"] = pd.to_datetime(
            d["UTCSTRING"].astype(str).str.strip().str.replace("Z", "", regex=False),
            utc=True,
            errors="coerce",
        )
    elif "TIME" in d.columns:
        epoch = pd.Timestamp("2017-01-01T00:00:00Z")
        d["_UTC"] = epoch + pd.to_timedelta(d["TIME"], unit="s")
    elif "Time" in d.columns:
        epoch = pd.Timestamp("2017-01-01T00:00:00Z")
        d["_UTC"] = epoch + pd.to_timedelta(d["Time"], unit="s")
    else:
        raise ValueError("No time column found. Expected UTC, UTCSTRING, TIME, or Time")
    return d


def filter_time(df_in: pd.DataFrame, start_utc: str, end_utc: str) -> pd.DataFrame:
    start_ts = pd.Timestamp(start_utc, tz="UTC")
    end_ts = pd.Timestamp(end_utc, tz="UTC")
    return df_in[(df_in["_UTC"] >= start_ts) & (df_in["_UTC"] <= end_ts)].copy()


# Build full dataframes from each dataset
df1_full = add_utc_column(fits_to_dataframe(file1))
df2_full = add_utc_column(fits_to_dataframe(file2))

# Create the two requested time-filtered datasets
df1_time_filtered = filter_time(df1_full, START_UTC, END_UTC)
df2_time_filtered = filter_time(df2_full, START_UTC_2, END_UTC_2)

print("Dataset 1 rows in window:", len(df1_time_filtered))
print("Dataset 2 rows in window:", len(df2_time_filtered))
print("Dataset 2 window:", START_UTC_2, "to", END_UTC_2)

print("\nDataset 1 preview:")
display(df1_time_filtered.head())

print("\nDataset 2 preview:")
display(df2_time_filtered.head())

Dataset 1 rows in window: 180
Dataset 2 rows in window: 96
Dataset 2 window: 2021-08-28T12:53:05 to 2021-08-28T12:54:41

Dataset 1 preview:


,Time,UTCString,FrameNumber,BDHTime,XSMTime,DecodingStatusFlag,_UTC
38398,1.075200e+08,2020-05-29 10:40:00.491824000,13736,2.697742e+07,971443.078044,0,2020-05-29 10:40:00.491824001+00:00
38399,1.075200e+08,2020-05-29 10:40:01.491879000,13737,2.697742e+07,971444.078044,0,2020-05-29 10:40:01.491879001+00:00
38400,1.075200e+08,2020-05-29 10:40:02.491929000,13738,2.697742e+07,971445.078044,0,2020-05-29 10:40:02.491928995+00:00
38401,1.075200e+08,2020-05-29 10:40:03.491969000,13739,2.697742e+07,971446.078044,0,2020-05-29 10:40:03.491968989+00:00
38402,1.075200e+08,2020-05-29 10:40:04.491899000,13740,2.697742e+07,971447.078044,0,2020-05-29 10:40:04.491899014+00:00



Dataset 2 preview:


,Time,UTCString,FrameNumber,BDHTime,XSMTime,DecodingStatusFlag,_UTC
46383,1.469264e+08,2021-08-28 12:53:05.519084000,5103,6.638381e+07,1.405148e+06,0,2021-08-28 12:53:05.519083977+00:00
46384,1.469264e+08,2021-08-28 12:53:06.519014000,5104,6.638381e+07,1.405149e+06,0,2021-08-28 12:53:06.519014031+00:00
46385,1.469264e+08,2021-08-28 12:53:07.519184000,5105,6.638381e+07,1.405150e+06,0,2021-08-28 12:53:07.519184023+00:00
46386,1.469264e+08,2021-08-28 12:53:08.519104000,5106,6.638381e+07,1.405151e+06,0,2021-08-28 12:53:08.519104004+00:00
46387,1.469264e+08,2021-08-28 12:53:09.519034000,5107,6.638381e+07,1.405152e+06,0,2021-08-28 12:53:09.519033998+00:00
